Model

In [18]:
import numpy as np

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label


class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples=5, min_samples_leaf=1, n_features=None, class_weights=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.min_samples_leaf = min_samples_leaf
        self.n_features = n_features
        self.class_weights = class_weights
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_features_total = X.shape[1]
        self.n_features = self.n_features or self.n_features_total
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]

        # stopping conditions
        if (
            depth >= self.max_depth or
            n_samples < self.min_samples or
            len(np.unique(y)) == 1
        ):
            return DecisionTreeNode(label=self._majority_label(y))
        np.random.seed(42)
        feature_idxs = np.random.choice(self.n_features_total, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            return DecisionTreeNode(label=self._majority_label(y))

        left_mask = X[:, best_feature] < best_threshold
        right_mask = ~left_mask

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left, right)

    def _best_split(self, X, y, feature_idxs):
        best_gini = float("inf")
        best_feature, best_threshold = None, None

        for feature in feature_idxs:
            X_col = X[:, feature]
            thresholds = np.unique(X_col)

            step = max(1, len(thresholds) // 10)

            for t in thresholds[::step]:
                gini = self._gini_split(y, X_col, t)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

    def _gini_split(self, y, X_col, threshold):
        left  = y[X_col <  threshold]
        right = y[X_col >= threshold]

        if len(left) < self.min_samples_leaf or len(right) < self.min_samples_leaf:
            return float("inf")

        def weighted_gini(group):
            if len(group) == 0:
                return 0
            classes, counts = np.unique(group, return_counts=True)
            if self.class_weights is not None:
                w = np.array([counts[i] * self.class_weights.get(classes[i], 1) for i in range(len(classes))])
            else:
                w = counts
            probs = w / w.sum()
            return 1 - np.sum(probs ** 2)

        def total_weight(group):
            if self.class_weights is None:
                return len(group)
            return sum(self.class_weights.get(c, 1) for c in group)

        w_left  = total_weight(left)
        w_right = total_weight(right)
        w_total = w_left + w_right

        return (w_left / w_total) * weighted_gini(left) + (w_right / w_total) * weighted_gini(right)

    def _majority_label(self, y):
        classes, counts = np.unique(y, return_counts=True)

        if self.class_weights is not None:
            weighted_counts = np.array([
                counts[i] * self.class_weights.get(classes[i], 1)
                for i in range(len(classes))
            ])
        else:
            weighted_counts = counts

        return classes[np.argmax(weighted_counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.label is not None:
            return node.label

        if x[node.feature] < node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

In [ ]:
import importlib
import preprocessing
importlib.reload(preprocessing)

HOG + PCA

In [8]:
from preprocessing import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score, k_fold_indices
import numpy as np
import time

X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="hog_pca", n_pca=50,balance = False)

folds = k_fold_indices(X_train, k=3)

fold_accuracies = []
all_y_true = []
all_y_pred = []

print("Running 3-Fold Cross-Validation...")
for fold_num, (train_idx, val_idx) in enumerate(folds):
    X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
    X_fold_val, y_fold_val = X_train[val_idx], y_train[val_idx]

    tree = CustomDecisionTree(
        max_depth=10,
        min_samples=10,
        n_features=X_fold_train.shape[1],
        class_weights=weights
    )

    start = time.time()
    tree.fit(X_fold_train, y_fold_train)
    elapsed = time.time() - start

    preds = tree.predict(X_fold_val)
    acc = custom_accuracy_score(y_fold_val, preds)
    fold_accuracies.append(acc)
    all_y_true.extend(y_fold_val)
    all_y_pred.extend(preds)

    print(f"  Fold {fold_num+1}: Accuracy={acc:.4f}  ({elapsed:.2f}s)")

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

print("\n" + "="*45)
print("  CUSTOM MODEL K-FOLD PERFORMANCE (HOG + PCA)")
print("="*45)

target_names = ["0 (Class 0)", "Not 0 (Class 1)"]
print(custom_classification_report(all_y_true, all_y_pred, target_names=target_names))
print(f"Mean Accuracy: {np.mean(fold_accuracies):.4f}  Std: {np.std(fold_accuracies):.4f}")

cm = custom_confusion_matrix(all_y_true, all_y_pred)
print("\nConfusion Matrix (rows = actual, cols = predicted):\n")

labels = ["0", "1"]
print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()
for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for v in row:
        print(f"{v:8}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Running 3-Fold Cross-Validation...
  Fold 1: Accuracy=0.9697  (19.66s)
  Fold 2: Accuracy=0.9724  (20.16s)
  Fold 3: Accuracy=0.9696  (19.10s)

  CUSTOM MODEL K-FOLD PERFORMANCE (HOG + PCA)
                 precision     recall   f1-score    support

0 (Class 0)           0.80       0.94       0.86       5336
Not 0 (Class 1)       0.99       0.97       0.98      48664

accuracy                                    0.97      54000
macro avg             0.90       0.96       0.92      54000

Mean Accuracy: 0.9705  Std: 0.0013

Confusion Matrix (rows = actual, cols = predicted):

                   0       1
         0     5003     333
         1     1258   47406


Grid Search Cross-Validation

In [ ]:

from preprocessing import (
    preprocess,
    custom_classification_report,
    custom_confusion_matrix,
    custom_accuracy_score
)
import numpy as np
import time


# ── Custom k-fold split ───────────────────────────────────────────────────────
def custom_k_fold(X, k=3, shuffle=True, seed=42):
    n = len(X)
    indices = np.arange(n)

    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    fold_sizes = np.full(k, n // k)
    fold_sizes[: n % k] += 1

    folds = []
    start = 0

    for size in fold_sizes:
        val_idx = indices[start: start + size]
        train_idx = np.concatenate([indices[:start], indices[start + size:]])
        folds.append((train_idx, val_idx))
        start += size

    return folds


# ── Parameter grid ────────────────────────────────────────────────────────────
param_grid = [
    (feature, balance, use_w, label)
    for feature in ["pca", "hog", "flatten"]
    for balance, use_w, label in [
        (False, True,  "weights"),
        (True,  False, "balanced"),
    ]
]

target_names = ["0 (Class 0)", "Not 0 (Class 1)"]
col_labels   = ["0", "1"]
SEP          = "=" * 55
results      = []


for feature, balance, use_w, weight_label in param_grid:

    X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(
        feature_method=feature, n_pca=50, balance=balance
    )

    class_weights = weights if use_w else None
    folds = custom_k_fold(X_train, k=3)

    fold_accuracies = []
    all_y_true = []
    all_y_pred = []

    combo_title = f"{feature.upper()} | {weight_label}"

    print(f"\n{SEP}")
    print(f"  {combo_title}")
    print(SEP)
    print("  Running 3-Fold Cross-Validation...\n")

    for fold_num, (train_idx, val_idx) in enumerate(folds):
        X_f_train, y_f_train = X_train[train_idx], y_train[train_idx]
        X_f_val,   y_f_val   = X_train[val_idx],   y_train[val_idx]

        tree = CustomDecisionTree(
            max_depth=10,
            min_samples=10,
            n_features=X_f_train.shape[1],
            class_weights=class_weights,
        )

        t0 = time.time()
        tree.fit(X_f_train, y_f_train)
        elapsed = time.time() - t0

        preds = tree.predict(X_f_val)
        acc = custom_accuracy_score(y_f_val, preds)

        fold_accuracies.append(acc)
        all_y_true.extend(y_f_val)
        all_y_pred.extend(preds)

    # ── Aggregate Results ─────────────────────────────────────────────────────
    all_y_true = np.array(all_y_true)
    all_y_pred = np.array(all_y_pred)

    mean_acc = np.mean(fold_accuracies)
    std_acc  = np.std(fold_accuracies)

    print(f"  ── Final Cross-Validation Report ({combo_title}) ──")
    print(custom_classification_report(all_y_true, all_y_pred, target_names=target_names))
    print(f"  Mean Accuracy : {mean_acc:.4f}   Std: {std_acc:.4f}")

    agg_cm = custom_confusion_matrix(all_y_true, all_y_pred)

    print("\n  Confusion Matrix (rows=actual, cols=predicted):\n")
    print(f"  {'':10}", end="")
    for cl in col_labels:
        print(f"{cl:>8}", end="")
    print()

    for i, row in enumerate(agg_cm):
        print(f"  {col_labels[i]:>10} ", end="")
        for v in row:
            print(f"{v:8}", end="")
        print()

    results.append({
        "feature": feature,
        "weight_config": weight_label,
        "mean_acc": mean_acc,
        "std_acc": std_acc,
        "fold_accs": fold_accuracies[:],
    })


# ── Best params summary ───────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  BEST PARAMS SUMMARY (sorted by mean accuracy)")
print(SEP)

results.sort(key=lambda r: r["mean_acc"], reverse=True)

print(f"  {'Rank':<5} {'Feature':<10} {'Weight Config':<25} {'Mean Acc':>9} {'Std':>7}  Fold Accs")
print(f"  {'-'*4} {'-'*9} {'-'*24} {'-'*9} {'-'*6}  {'-'*30}")

for rank, r in enumerate(results, 1):
    fold_str = "  ".join(f"{a:.4f}" for a in r["fold_accs"])
    print(f"  {rank:<5} {r['feature']:<10} {r['weight_config']:<25} {r['mean_acc']:>9.4f} {r['std_acc']:>7.4f}  [{fold_str}]")

best = results[0]

print(f"\n  >> Best combination:")
print(f"     Feature       : {best['feature']}")
print(f"     Weight config : {best['weight_config']}")
print(f"     Mean Accuracy : {best['mean_acc']:.4f}  (std={best['std_acc']:.4f})")



Best Model(Flat with weights) with Test Data score

In [ ]:
from preprocessing import (
    preprocess,
    custom_classification_report,
    custom_confusion_matrix,
    custom_accuracy_score
)
import numpy as np
import time

X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(
    feature_method="flatten",
    n_pca=50,
    balance=False
)

tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],
    class_weights=weights
)

start = time.time()
tree.fit(X_train, y_train)
elapsed = time.time() - start

preds = tree.predict(X_test)
acc = custom_accuracy_score(y_test, preds)

print(f"\nTest Accuracy: {acc:.4f}  ({elapsed:.2f}s)")

print("\n" + "="*45)
print("  CUSTOM MODEL TEST PERFORMANCE (FLATTEN)")
print("="*45)

target_names = ["0 (Class 0)", "Not 0 (Class 1)"]
print(custom_classification_report(y_test, preds, target_names=target_names))

cm = custom_confusion_matrix(y_test, preds)
print("\nConfusion Matrix (rows = actual, cols = predicted):\n")

labels = ["0", "1"]
print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for v in row:
        print(f"{v:8}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000

Test Accuracy: 0.9784  (361.71s)

  CUSTOM MODEL TEST PERFORMANCE (HOG)
                 precision     recall   f1-score    support

0 (Class 0)           0.84       0.97       0.90        980
Not 0 (Class 1)       1.00       0.98       0.99       9020

accuracy                                    0.98      10000
macro avg             0.92       0.97       0.94      10000


Confusion Matrix (rows = actual, cols = predicted):

                   0       1
         0      948      32
         1      184    8836
